In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "06-gateway/identity-security/agentic-identity-gcp-lab/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 07 · A2A: Agent Cards and identity between agents

**Primer section:** §7.2 A2A — identity between agents · threat model row ASI07 (insecure
inter-agent communication), ASI04 (supply chain).

In A2A the **Agent Card** is the discovery document a client fetches before talking to an agent. Two
parts matter for security: `securitySchemes` + `security` requirements declare *how* a caller must
authenticate and which scopes a skill needs; **signatures** (`AgentCardSignature`, a detached JWS)
let a client verify the card was not tampered with in a registry or in transit. Then the design
rules for multi-agent systems: every hop is a fresh authorization decision, and delegation is
*exchanged*, never *forwarded*.

In [ ]:
from agentsec.logging_utils import quiet_logs

quiet_logs()

import json

import jwt  # display only

from agentsec.a2a import (
    A2AAuthError,
    authorize_inbound,
    build_agent_card,
    card_from_dict,
    card_to_dict,
    required_scopes,
    sign_agent_card,
    token_for_peer,
    verify_agent_card,
)
from agentsec.identity import AgentIdentity, LocalRuntimeCA, TokenIssuer, UserPrincipal


def peek(token: str) -> dict:
    return jwt.decode(token, options={"verify_signature": False})

def short(spiffe: str) -> str:
    return spiffe.rsplit("/", 1)[-1]

ORG, PROJECT = "123456789012", "987654321098"
support = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="support-agent", org_id=ORG)
refunds = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="refunds-agent", org_id=ORG)
ana = UserPrincipal(subject="u-ana", email="ana@customer.example", tenant="acme")

issuer = TokenIssuer()
ca = LocalRuntimeCA()
support_cert = ca.issue(support)

SUPPORT_URL = "https://agents.acme.example/support"
REFUNDS_URL = "https://agents.acme.example/refunds"
FRONT_END = "https://app.acme.example"

## 1. Build a card that declares its security schemes

The reference card requires a bearer JWT from our STS with scope `agent:invoke` and also advertises
mutual TLS with a runtime-issued SPIFFE certificate. `card_to_dict` is what a registry stores and
what a client fetches from `/.well-known/agent-card.json`.

In [ ]:
card = build_agent_card(agent=support, name="support", url=SUPPORT_URL, issuer=issuer.issuer)
doc = card_to_dict(card)
print(json.dumps({k: doc[k] for k in ("name", "supportedInterfaces", "securitySchemes", "securityRequirements")}, indent=2))
print("skills:", [(s["id"], s.get("securityRequirements")) for s in doc["skills"]])
print("required scopes (card):", required_scopes(card), "| for skill 'support':", required_scopes(card, "support"))
assert required_scopes(card) == {"agent:invoke"}

## 2. Sign, publish, verify — and detect tampering

The card is signed with the agent's certificate key (the key behind its SPIFFE identity), `kid` = the
certificate thumbprint. A client verifies against the keys it trusts (a JWKS or the registry's key
set). A single changed byte — say, the interface URL pointing at an attacker's endpoint — invalidates
the signature.

In [ ]:
signed = sign_agent_card(card, support_cert.private_key, kid=support_cert.thumbprint)
trusted_keys = {support_cert.thumbprint: support_cert.private_key.public_key()}
print("signatures on card:", len(signed.signatures), "| protected header:", json.loads(jwt.utils.base64url_decode(signed.signatures[0].protected)))
print("verified with kid :", verify_agent_card(signed, trusted_keys)[:16], "…")

# Round trip through JSON (what a registry stores) still verifies
stored = card_to_dict(signed)
assert verify_agent_card(card_from_dict(stored), trusted_keys) == support_cert.thumbprint

def check(label, doc_or_card):
    try:
        verify_agent_card(doc_or_card, trusted_keys)
        print(f"{label:<44} verified")
    except Exception as e:
        print(f"{label:<44} REJECTED {type(e).__name__}{': ' + str(e)[:60] if str(e) else ' (signature does not verify)'}")

tampered = json.loads(json.dumps(stored))
tampered["supportedInterfaces"][0]["url"] = "https://evil.example/support"
check("tampered interface URL", card_from_dict(tampered))
check("unsigned card", card)
check("signed by an unknown key", sign_agent_card(card, ca.issue(refunds).private_key, kid="unknown-kid"))
check("original signed card", signed)

## 3. Calling a peer: exchange, never forward

The support agent holds Ana's delegated token (audience = the front-end). To call the refunds agent it
obtains a token **for the peer's audience** via RFC 8693 exchange with itself as actor
(`token_for_peer`). The user's consent propagates; the calling agent is recorded as the actor; scopes
are narrowed to what the peer's card requires.

In [ ]:
user_token = issuer.mint(subject=ana.subject, audience=FRONT_END, scope="agent:invoke tickets:read tickets:write", extra={"email": ana.email})

peer_token = token_for_peer(issuer, caller=support, current_token=user_token, peer_audience=REFUNDS_URL)
claims = peek(peer_token)
print("token for refunds-agent:", {"sub": claims["sub"], "aud": claims["aud"], "scope": claims["scope"], "act": short(claims["act"]["sub"]), "authority": claims["authority"]})
assert claims["aud"] == REFUNDS_URL and claims["scope"] == "agent:invoke" and claims["act"]["sub"] == support.spiffe_id

## 4. The receiving agent re-authorizes every hop

`authorize_inbound` is what the refunds agent runs on each request: verify the bearer against **its
own** audience, check the scopes its card advertised, cap the delegation depth, and build the
`AuthorityContext` it will act under — with itself appended to the chain.

In [ ]:
claims, authority = authorize_inbound({"Authorization": f"Bearer {peer_token}"}, issuer=issuer, audience=REFUNDS_URL, this_agent=refunds)
print("user      :", authority.user.email)
print("authority :", authority.mode.value, "| scopes:", sorted(authority.scopes))
print("hop chain :", [short(a) for a in authority.chain])
print("audit ids :", {k: (short(v) if k == "agent" else v) for k, v in authority.audit_identities().items()})
assert authority.chain == (refunds.spiffe_id, support.spiffe_id) and authority.scopes == {"agent:invoke"}

### What gets rejected

A **forwarded** user token (audience = the front-end) fails at the peer — wrong audience. A token
without the card's required scope fails with `insufficient_scope`. A chain deeper than the configured
maximum fails. And the own-authority path (no user) works but carries no user identity.

In [ ]:
def inbound(label, token, **kw):
    try:
        _, auth = authorize_inbound({"Authorization": f"Bearer {token}"}, issuer=issuer, audience=REFUNDS_URL, this_agent=refunds, **kw)
        print(f"{label:<44} accepted  authority={auth.mode.value} user={auth.user.email if auth.user else None}")
    except A2AAuthError as e:
        print(f"{label:<44} REJECTED  {e}")

inbound("exchanged token (correct)", peer_token)
inbound("forwarded user token (aud = front-end)", user_token)
no_scope = issuer.mint(subject=ana.subject, audience=FRONT_END, scope="tickets:read", extra={"email": ana.email})
inbound("exchanged from a token lacking agent:invoke", token_for_peer(issuer, caller=support, current_token=no_scope, peer_audience=REFUNDS_URL))
inbound("delegation chain deeper than allowed", peer_token, max_delegation_depth=0)
own_token = token_for_peer(issuer, caller=support, current_token=None, peer_audience=REFUNDS_URL)
inbound("support-agent under its own authority", own_token)

for bad in (user_token,):
    try:
        authorize_inbound({"Authorization": f"Bearer {bad}"}, issuer=issuer, audience=REFUNDS_URL, this_agent=refunds)
        raise AssertionError("forwarded token must be rejected")
    except A2AAuthError:
        pass

## 5. Two hops: the refunds agent calls a payments agent

The refunds agent does not forward the token it received either. It exchanges again; the payments
agent sees the full chain (`act.act`), still with Ana as subject.

In [ ]:
payments = AgentIdentity.for_agent_engine(project_number=PROJECT, location="us-central1", engine_id="payments-agent", org_id=ORG)
PAYMENTS_URL = "https://agents.acme.example/payments"
hop2_token = token_for_peer(issuer, caller=refunds, current_token=peer_token, peer_audience=PAYMENTS_URL)
claims2, authority2 = authorize_inbound({"Authorization": f"Bearer {hop2_token}"}, issuer=issuer, audience=PAYMENTS_URL, this_agent=payments)
print("subject   :", claims2.subject, "| actor chain:", [short(a) for a in claims2.actor_chain])
print("hop chain :", [short(a) for a in authority2.chain])
assert claims2.actor_chain == [refunds.spiffe_id, support.spiffe_id]
assert authority2.chain == (payments.spiffe_id, refunds.spiffe_id, support.spiffe_id)

## On Google Cloud

* Register peer agents in **Agent Registry**; the gateway only routes to registered destinations, so
  the registry is the supply-chain allowlist. Prefer signed cards.
* Each agent has its own Agent Identity; A2A traffic through Agent Gateway is mTLS + DPoP bound and
  IAM-authorized per SPIFFE ID; tasks can enter `TASK_STATE_AUTH_REQUIRED` to request credentials.
* Budget and depth limits on agent-to-agent recursion (`max_delegation_depth`) bound cascading failures.

**In one sentence:** "For A2A the card declares the security schemes and the scopes per skill,
and it is signed so a registry cannot be poisoned silently. Every hop re-authorizes against its own
audience, and delegation is exchanged rather than forwarded, so the user's consent propagates with an
explicit actor chain and the agent's own authority never silently expands."